# 1장 — 큰 그림: 숫자의 몸무게 ⚖️

웹 챕터의 두 실험 — 부품별 저울과 몸무게 계산기 — 를 **직접 코드로** 돌려 봅니다.
이번엔 진짜 행렬을 만들어 개수를 세고, 실제 바이트 수까지 잽니다.

> **노트북 사용법**: 각 셀(회색 상자)을 클릭하고 `Shift + Enter` — 위에서 아래로 반복하면 끝.

**이 노트북의 목표**: 9부에서 가장 자주 쓸 한 줄을 손에 넣는다 —
**"파일 크기 = 파라미터 개수 × 숫자 하나의 칸 크기."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch1')

## 1. 부품별 저울 — 근사 공식 세 줄

4부 10장에서 조립한 구조 그대로입니다: 임베딩 표(V×d) + 블록 L개(어텐션 Q·K·V·O 4d² + FFN 8d²).
bias·LayerNorm 같은 티끌은 생략한 **근사**라는 걸 기억하세요 — 얼마나 좋은 근사인지 바로 검증합니다.

In [ ]:
def count_params(V, d, L):
    """미니 GPT 몸무게 공식(근사) — 임베딩 V·d + 블록 L·(어텐션 4d² + FFN 8d²)."""
    return V * d + L * 12 * d**2

# 검증 ① — GPT-2 소형 (V=50,257 / d=768 / L=12): 실제 공개 값은 124M
gpt2 = count_params(50257, 768, 12)
print(f'GPT-2 소형 추정: {gpt2:,} 개  ≈ {gpt2 / 1e6:.1f}M')

# 검증 ② — 7B급 (V=32,000 / d=4096 / L=32)
seven = count_params(32000, 4096, 32)
print(f'7B급 추정:      {seven:,} 개  ≈ {seven / 1e9:.2f}B')

assert abs(gpt2 - 124e6) / 124e6 < 0.01, '공개 값 124M 과 오차 1% 미만 — 근사 공식이 몸통을 다 담았다'
assert 6.4e9 < seven < 7.0e9, '시중 7B 모델의 정체 — 근사 공식으로 6.57B'
print()
print('✔ 확인: 스펙표의 B·M 은 신비한 숫자가 아니라 부품별 곱셈의 합이다')

## 2. 말고 진짜로 세보기 — 미니 모델을 만들어 저울에 올린다

공식이 아니라 **실제 행렬 덩어리**를 만들어 `size`(개수)와 `nbytes`(바이트)를 재 봅니다.

In [ ]:
import numpy as np

V, d, L = 1000, 64, 4                      # 미니 프리셋 (웹 실험 ①과 동일)
rng = np.random.default_rng(9)              # 시드 고정 — 결정적

model = {'embedding': rng.standard_normal((V, d)).astype(np.float32)}
for i in range(L):
    model[f'block{i}.attn_qkvo'] = rng.standard_normal((4, d, d)).astype(np.float32)
    model[f'block{i}.ffn_up'] = rng.standard_normal((d, 4 * d)).astype(np.float32)
    model[f'block{i}.ffn_down'] = rng.standard_normal((4 * d, d)).astype(np.float32)

total = sum(w.size for w in model.values())
nbytes = sum(w.nbytes for w in model.values())
print(f'행렬 수 {len(model)} 개 · 파라미터 {total:,} 개 · fp32 로 {nbytes:,} 바이트 (= {nbytes / 1e6:.2f}MB)')

assert total == count_params(V, d, L), '진짜로 센 개수 = 근사 공식 (미니는 생략 부품이 없어 정확히 일치)'
assert nbytes == total * 4, 'fp32 = 숫자 하나에 4바이트 — 파일 크기 공식의 실측 확인'
print()
print('✔ 확인: 파일 크기 = 개수 × 칸 크기 — 공식이 아니라 실측이다')

## 3. 몸무게 계산기 — 같은 7B, 다른 칸

딱 70억 개라 치고, 칸 크기만 바꿔 봅니다 — 같은 모델의 파일 크기가 왜 제각각인지.

In [ ]:
n = 7_000_000_000                     # 딱 70억 개라 치자
print(' 칸(정밀도)   숫자 하나   파일 크기(십진 GB)')
for name, size in [('fp32', 4), ('fp16', 2), ('int8', 1), ('int4', 0.5)]:
    print(f'   {name:<6s}    {size:>4} 바이트     {n * size / 1e9:5.1f} GB')

assert n * 2 / 1e9 == 14.0, '7B × fp16 = 14GB — 모델 카드에서 봤을 그 크기'
assert n * 0.5 / 1e9 == 3.5, '7B × int4 = 3.5GB — Ollama 태그 q4 가 국민 태그인 이유'
print()
print('✔ 확인: 개수는 그대로, 칸만 좁혀서 26.3GB → 3.5GB — 무엇을 잃는 대가인지는 8~10장에서 측정')
print('  (참고: 파일 관리자는 GiB(2^30 바이트) 자를 써서 14GB 가 13.0GiB 로 보인다 — 둘 다 맞는 값)')

**직접 해보기**: ① `count_params` 에 d=8192, L=80, V=32000 을 넣으면? (70B급의 정체)
② 여러분 PC의 RAM 용량으로 fp16 / int4 중 어느 칸의 7B가 들어갈지 계산해 보세요 —
가중치 몫만이고, 문맥(KV 캐시) 몫까지 정산하는 법은 11장 캡스톤에서.

## 4. 관찰 정리 + 🤖 AI 연결

- **7B 의 정체**: 파라미터(가중치) 약 70억 개 — 임베딩 표 V·d + 블록 L·12d², 몸통은 **d² 행렬들**
  (d 2배 → 블록 4배). GPT-2 소형로 검산하니 공개 값 124M 과 1% 미만 오차.
- **파일 크기 = 개수 × 칸 크기** — 같은 7B 가 fp16 이면 14GB, int4 면 3.5GB.
  체크포인트(모델 파일) = 파라미터 숫자들을 통째로 담은 것.
- **회수하는 수학**: 4부 10장(전체 조립 — 같은 구조를 개수로 재독) ·
  6부 8장(LoRA 계산기 — 그때는 "몇 개를 새로 배우나", 오늘은 "몇 바이트에 담나").

**다음 챕터**: 2장 — 컴퓨터의 손가락. "칸"의 재료인 비트부터 —
스위치 n개로 몇 가지를 셀 수 있는지, 음수는 어떻게 담는지.